# 06 - Power BI Ready Views

## Purpose

Create and validate business-ready reporting views for Power BI consumption.

## Concepts covered

- Star schema joins for reporting
- Transaction detail view
- Monthly summary view
- Customer 360 view
- Branch activity view
- Product usage view
- Executive KPI view
- Power BI-friendly naming and validation

## Prerequisites

03_gold_dimensional_model.ipynb completed successfully and Gold Delta tables exist.

## Expected output

Spark views for notebook validation, example DataFrames matching the SQL scripts, and guidance for persistent SQL Analytics Endpoint views.

In [ ]:
from datetime import datetime, timezone
from typing import Dict, List

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

REQUIRED_TABLES = ["fact_transaction", "dim_date", "dim_customer", "dim_account", "dim_product", "dim_branch"]

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def require_table(table_name: str) -> DataFrame:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required table is missing: {table_name}")
    row_count = spark.table(table_name).count()
    log_info(f"{table_name}: {row_count} rows")
    return spark.table(table_name)

for table_name in REQUIRED_TABLES:
    require_table(table_name)

## Create reusable Spark views

These temporary Spark views let you validate the reporting shapes from the notebook. Use the SQL scripts in the repository to create persistent SQL Analytics Endpoint views.

In [ ]:
VIEW_SQL: Dict[str, str] = {
    "vw_powerbi_transaction_detail": """
        SELECT
            f.transaction_id,
            d.date_value AS transaction_date,
            d.year,
            d.quarter,
            d.month,
            d.month_name,
            d.year_month,
            c.customer_id,
            c.customer_full_name,
            c.customer_segment,
            c.customer_status,
            c.city AS customer_city,
            c.state AS customer_state,
            a.account_id,
            a.account_type,
            a.account_status,
            a.current_balance,
            p.product_id,
            p.product_name,
            p.product_category,
            p.product_family,
            b.branch_id,
            b.branch_name,
            b.region,
            f.transaction_channel,
            f.transaction_type,
            f.transaction_status,
            f.currency,
            f.transaction_amount,
            f.absolute_transaction_amount,
            f.debit_credit_indicator,
            f.is_posted,
            f.transaction_count
        FROM fact_transaction f
        JOIN dim_date d ON f.date_key = d.date_key
        JOIN dim_customer c ON f.customer_key = c.customer_key
        JOIN dim_account a ON f.account_key = a.account_key
        JOIN dim_product p ON f.product_key = p.product_key
        JOIN dim_branch b ON f.branch_key = b.branch_key
    """,
    "vw_powerbi_monthly_transaction_summary": """
        SELECT
            d.year_month,
            d.year,
            d.month,
            p.product_category,
            p.product_family,
            b.region,
            COUNT(DISTINCT f.transaction_id) AS transaction_count,
            SUM(f.absolute_transaction_amount) AS total_transaction_amount,
            AVG(f.absolute_transaction_amount) AS average_transaction_amount,
            SUM(CASE WHEN f.transaction_amount > 0 THEN f.transaction_amount ELSE 0 END) AS total_credit_amount,
            SUM(CASE WHEN f.transaction_amount < 0 THEN ABS(f.transaction_amount) ELSE 0 END) AS total_debit_amount
        FROM fact_transaction f
        JOIN dim_date d ON f.date_key = d.date_key
        JOIN dim_product p ON f.product_key = p.product_key
        JOIN dim_branch b ON f.branch_key = b.branch_key
        WHERE f.is_posted = true
        GROUP BY d.year_month, d.year, d.month, p.product_category, p.product_family, b.region
    """,
    "vw_powerbi_customer_360": """
        WITH account_summary AS (
            SELECT
                customer_id,
                COUNT(DISTINCT account_id) AS account_count,
                SUM(current_balance) AS total_current_balance
            FROM dim_account
            GROUP BY customer_id
        ),
        transaction_summary AS (
            SELECT
                customer_key,
                COUNT(DISTINCT transaction_id) AS transaction_count,
                SUM(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE 0 END) AS posted_transaction_amount,
                MAX(transaction_timestamp) AS last_transaction_timestamp
            FROM fact_transaction
            GROUP BY customer_key
        )
        SELECT
            c.customer_key,
            c.customer_id,
            c.customer_full_name,
            c.customer_segment,
            c.customer_status,
            c.city,
            c.state,
            c.customer_age_years,
            c.customer_tenure_years,
            COALESCE(a.account_count, 0) AS account_count,
            COALESCE(a.total_current_balance, 0) AS total_current_balance,
            COALESCE(t.transaction_count, 0) AS transaction_count,
            COALESCE(t.posted_transaction_amount, 0) AS posted_transaction_amount,
            t.last_transaction_timestamp
        FROM dim_customer c
        LEFT JOIN account_summary a ON c.customer_id = a.customer_id
        LEFT JOIN transaction_summary t ON c.customer_key = t.customer_key
    """,
    "vw_powerbi_branch_activity": """
        WITH account_summary AS (
            SELECT branch_id, COUNT(DISTINCT account_id) AS account_count
            FROM dim_account
            GROUP BY branch_id
        ),
        transaction_summary AS (
            SELECT
                branch_key,
                COUNT(DISTINCT transaction_id) AS transaction_count,
                SUM(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE 0 END) AS posted_transaction_amount,
                AVG(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE NULL END) AS average_posted_transaction_amount
            FROM fact_transaction
            GROUP BY branch_key
        )
        SELECT
            b.branch_key,
            b.branch_id,
            b.branch_name,
            b.region,
            b.city,
            b.state,
            COALESCE(a.account_count, 0) AS account_count,
            COALESCE(t.transaction_count, 0) AS transaction_count,
            COALESCE(t.posted_transaction_amount, 0) AS posted_transaction_amount,
            t.average_posted_transaction_amount
        FROM dim_branch b
        LEFT JOIN account_summary a ON b.branch_id = a.branch_id
        LEFT JOIN transaction_summary t ON b.branch_key = t.branch_key
    """,
    "vw_powerbi_product_usage": """
        WITH account_summary AS (
            SELECT product_id, COUNT(DISTINCT account_id) AS account_count
            FROM dim_account
            GROUP BY product_id
        ),
        transaction_summary AS (
            SELECT
                product_key,
                COUNT(DISTINCT transaction_id) AS transaction_count,
                SUM(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE 0 END) AS posted_transaction_amount
            FROM fact_transaction
            GROUP BY product_key
        )
        SELECT
            p.product_key,
            p.product_id,
            p.product_name,
            p.product_category,
            p.product_family,
            p.fee_model,
            COALESCE(a.account_count, 0) AS account_count,
            COALESCE(t.transaction_count, 0) AS transaction_count,
            COALESCE(t.posted_transaction_amount, 0) AS posted_transaction_amount
        FROM dim_product p
        LEFT JOIN account_summary a ON p.product_id = a.product_id
        LEFT JOIN transaction_summary t ON p.product_key = t.product_key
    """,
    "vw_powerbi_executive_kpis": """
        WITH customer_summary AS (
            SELECT COUNT(DISTINCT CASE WHEN is_active_customer = true THEN customer_key END) AS active_customer_count
            FROM dim_customer
        ),
        account_summary AS (
            SELECT COUNT(DISTINCT account_key) AS account_count, SUM(current_balance) AS total_current_balance
            FROM dim_account
        ),
        transaction_summary AS (
            SELECT
                COUNT(DISTINCT transaction_id) AS transaction_count,
                SUM(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE 0 END) AS posted_transaction_amount,
                AVG(CASE WHEN is_posted = true THEN absolute_transaction_amount ELSE NULL END) AS average_posted_transaction_amount
            FROM fact_transaction
        )
        SELECT
            c.active_customer_count,
            a.account_count,
            t.transaction_count,
            t.posted_transaction_amount,
            t.average_posted_transaction_amount,
            a.total_current_balance
        FROM customer_summary c
        CROSS JOIN account_summary a
        CROSS JOIN transaction_summary t
    """,
}

def create_spark_view(view_name: str, sql_body: str) -> None:
    spark.sql(f"CREATE OR REPLACE TEMP VIEW {view_name} AS {sql_body}")
    log_info(f"Created temporary Spark view: {view_name}")

for view_name, sql_body in VIEW_SQL.items():
    create_spark_view(view_name, sql_body)

## Validate reporting shapes

In [ ]:
view_validation_rows: List[Dict[str, object]] = []

for view_name in VIEW_SQL:
    df = spark.table(view_name)
    row_count = df.count()
    column_count = len(df.columns)
    view_validation_rows.append({"view_name": view_name, "row_count": row_count, "column_count": column_count, "status": "PASS" if row_count > 0 else "REVIEW"})

view_validation_df = spark.createDataFrame(view_validation_rows)
display(view_validation_df.orderBy("view_name"))

## Preview Power BI-ready outputs

In [ ]:
display(spark.table("vw_powerbi_transaction_detail").orderBy("transaction_date", "transaction_id"))
display(spark.table("vw_powerbi_monthly_transaction_summary").orderBy("year_month", "product_category", "region"))
display(spark.table("vw_powerbi_customer_360").orderBy(F.desc("posted_transaction_amount")))
display(spark.table("vw_powerbi_branch_activity").orderBy(F.desc("posted_transaction_amount")))
display(spark.table("vw_powerbi_product_usage").orderBy(F.desc("posted_transaction_amount")))
display(spark.table("vw_powerbi_executive_kpis"))

## Business metric validation

In [ ]:
active_customer_count = spark.table("dim_customer").filter(F.col("is_active_customer") == True).select("customer_key").distinct().count()
kpi_active_customer_count = spark.table("vw_powerbi_executive_kpis").collect()[0]["active_customer_count"]

log_info(f"Active customers from dimension: {active_customer_count}")
log_info(f"Active customers from KPI view: {kpi_active_customer_count}")

if active_customer_count != kpi_active_customer_count:
    raise RuntimeError("Active customer count mismatch between dim_customer and executive KPI view")

transaction_detail_count = spark.table("vw_powerbi_transaction_detail").select("transaction_id").distinct().count()
fact_count = spark.table("fact_transaction").select("transaction_id").distinct().count()

if transaction_detail_count != fact_count:
    raise RuntimeError("Transaction detail view does not reconcile to fact_transaction")

log_info("Power BI reporting views validated successfully")

## Next steps

- Use sql/create_gold_views.sql and sql/powerbi_consumption_views.sql in the SQL Analytics Endpoint for persistent SQL views.
- In Power BI, prefer the Gold star schema for governed semantic models.
- Use the wide detail and summary views for quick exploration, validation, and beginner demos.